# NB01: Data Collection

The research aim for this project is to identify whether the winner of a professional tennis match is predicted by hitting more winners (aggression) or by committing fewer unforced errors (consistency). 

This notebook collects data from the `get_tournaments` and `get_fixtures` endpoints from API-Tennis.

## Setup

In [2]:
import os
import json
import requests
from dotenv import load_dotenv

Loading the API key (replace the placeholder in the .env with your own key):

In [3]:
load_dotenv()

API_KEY = os.getenv("OPENWEATHER_API_KEY")

## Context

### Tournaments

For the project, the tournaments chosen were the 3 Grand Slams completed thus far this year (as of July 2026). Only the men's singles draws were considered, as restricting to one tour only removes the need to control for differing tournament formats and player playstyles. 

Grand Slams were chosen deliberately for three reasons:

1. **Larger sample size**: Draws for Grand Slams contain more players than lower tier ATP tour events and thus more data points to investigate.
2. **Data completeness**: `statistics` are recorded much more consistently and reliably for high-profile tour events.
3. **Surface variety**: The three slams played so far (Australian Open, Roland-Garros, and Wimbledon) coincide with the three different playing surfaces (Hard, clay, grass respectively) which allows for further analysis on whether insights change depending on the surface the match is played on.


First we have to obtain the tournament keys for the tournaments of interest:

In [16]:
API_KEY = os.getenv("API_KEY")

url = "https://api.api-tennis.com/tennis/"
params = {
    "method": "get_tournaments",
    "APIkey": API_KEY
}

response = requests.get(url, params=params)
tournaments = response.json()["result"]

print(f"Status code: {response.status_code}")
print(f"Number of tournaments: {len(tournaments)}")

slam_names = ["Australian Open", "French Open", "Wimbledon"]

slam_tournaments = [t for t in tournaments if (t["tournament_name"] in slam_names) & ("Atp Singles" in t["event_type_type"])]

Status code: 200
Number of tournaments: 10154


In [17]:
slam_tournaments

[{'tournament_key': 1236,
  'tournament_name': 'Australian Open',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Hard'},
 {'tournament_key': 2155,
  'tournament_name': 'French Open',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Clay'},
 {'tournament_key': 2053,
  'tournament_name': 'Wimbledon',
  'event_type_key': 265,
  'event_type_type': 'Atp Singles',
  'tournament_sourface': 'Grass'}]

Now that we have the `tournament_key` for each Slam, we can work on obtaining the `statistics` field from the `get_fixtures` endpoint.

In [53]:
START_DATE = "2026-01-01"
STOP_DATE = "2026-07-31"

params = {
    "method": "get_fixtures",
    "APIkey": API_KEY,
    "date_start": START_DATE,
    "date_stop": STOP_DATE,
    "event_type_key": 265,
    "tournament_key": 2053,
}   

response = requests.get(url, params=params)
placeholder = response.json()

full_results = placeholder["result"]

fixture1 = full_results[0]

print(fixture1)
print(type(fixture1))


{'event_key': 12139093, 'event_date': '2026-06-22', 'event_time': '12:05', 'event_first_player': 'A. Andrade', 'first_player_key': 6505, 'event_second_player': 'C. Smith', 'second_player_key': 54466, 'event_final_result': '1 - 2', 'event_game_result': '-', 'event_serve': None, 'event_winner': 'Second Player', 'event_status': 'Finished', 'event_type_type': 'Atp Singles', 'tournament_name': 'Wimbledon', 'tournament_key': 2053, 'tournament_round': 'ATP Wimbledon - Quarter-finals', 'tournament_season': '2026', 'event_live': '0', 'event_first_player_logo': 'https://api.api-tennis.com/logo-tennis/6505_a-andrade.jpg', 'event_second_player_logo': 'https://api.api-tennis.com/logo-tennis/54466_c-smith.jpg', 'event_qualification': 'True', 'pointbypoint': [{'set_number': 'Set 1', 'number_game': '1', 'player_served': 'Second Player', 'serve_winner': 'Second Player', 'serve_lost': None, 'score': '0 - 1', 'points': [{'number_point': '1', 'score': '0 - 15', 'break_point': None, 'set_point': None, 'mat

In [60]:
to_keep = ["first_player_key", "second_player_key", "event_winner", "tournament_name", "statistics"]
fixturecopy = fixture1.copy()


fixtureedited = {key: val for key, val in fixturecopy.items() if key in to_keep}

print(fixtureedited)

{'first_player_key': 6505, 'second_player_key': 54466, 'event_winner': 'Second Player', 'tournament_name': 'Wimbledon', 'statistics': [{'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', 'stat_name': 'Aces', 'stat_value': '4', 'stat_won': None, 'stat_total': None}, {'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', 'stat_name': 'Double Faults', 'stat_value': '4', 'stat_won': None, 'stat_total': None}, {'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', 'stat_name': '1st serve percentage', 'stat_value': '59%', 'stat_won': None, 'stat_total': None}, {'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', 'stat_name': '1st serve points won', 'stat_value': '68%', 'stat_won': 32, 'stat_total': 47}, {'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', 'stat_name': '2nd serve points won', 'stat_value': '41%', 'stat_won': 13, 'stat_total': 32}, {'player_key': 6505, 'stat_period': 'match', 'stat_type': 'Service', '

### Variables of interest

Identifying fields (player keys, match winners, tournament name) and the full statistics list are kept from each match, from which NB02 will extract more required variables more precisely.
